# Gradient Boosting: From First Principles

Gradient Boosting is an additive ensemble method that builds weak learners sequentially. Each new learner is chosen to reduce a specified loss given the predictions made by the ensemble so far.

The name describes the method: it performs gradient-descent-like optimization over functions, adding a learner in a direction that locally decreases the loss. Decision trees are common base learners, but the central idea is the loss-gradient update, not a particular tree implementation.

## Why Gradient Boosting Is Useful

A single shallow tree is often too simple to capture a complicated relationship. Gradient Boosting combines many such learners; later learners are fit after earlier contributions have already changed the predictions.

This staged approach differs from Random Forest, where trees are generally trained independently and their predictions are averaged or voted. A Gradient Boosting ensemble can reduce bias by adding corrections, but its sequential fitting and flexible trees can also overfit if the number and complexity of learners are not controlled.

The loss function determines what counts as an error and therefore what each next learner tries to correct. This allows the same framework to support regression, classification, and other differentiable prediction objectives.

## Residuals and Pseudo-Residuals

For squared-error regression with loss $L(y,F)=\frac{1}{2}(y-F)^2$, the negative gradient is:

$$
-\frac{\partial L(y,F)}{\partial F}=y-F
$$

This is the ordinary residual: actual value minus current prediction. Fitting a new learner to these residuals therefore makes sense for squared-error regression.

For a general differentiable loss, the corresponding negative gradients are called pseudo-residuals. They measure a local direction to reduce loss, but they need not equal the raw target-minus-prediction difference. For example, with binary logistic loss and probability $p=\sigma(F)$, the negative gradient with respect to the raw score is $y-p$.

The term "gradient" is important: the next learner approximates a direction of improvement in prediction space. It does not necessarily directly predict the final target values.

## One Boosting Round

A common gradient-boosting procedure can be written as follows:

1. Start with a constant prediction that minimizes training loss: $F_0(x)=\underset{c}{\operatorname{argmin}}\sum_i L(y_i,c)$. For squared-error regression, this is the mean target value.
2. Compute each example's negative loss gradient at the current prediction; these are the pseudo-residuals.
3. Fit a weak learner, often a shallow regression tree, to predict the pseudo-residuals from the features.
4. Choose how much of that learner to add by minimizing the loss along the update direction, or by selecting leaf values that reduce the objective.
5. Update the ensemble with a shrunken step: $F_m(x)=F_{m-1}(x)+\eta\,\gamma_m h_m(x)$, where $\gamma_m$ is a fitted step size and $0<\eta\leq1$ is the learning rate.

Repeat the process for later rounds. Some descriptions absorb $\gamma_m$ into $h_m$; this is why the simpler additive formula may omit it.

## Worked Example: A Residual-Fitting Step

Suppose three observations have features $x=(1,2,3)$ and targets $y=(2,3,6)$. With squared-error loss, the initial constant model is the target mean, $F_0=11/3$. Its residuals are $-5/3$, $-2/3$, and $7/3$.

Fit a one-split regression tree with regions $x\leq2$ and $x>2$. The first leaf predicts the mean residual of its two observations, $-7/6$; the second predicts $7/3$. With learning rate $\eta=1/2$, the updated predictions are:

$$
F_1(1)=F_1(2)=\frac{11}{3}+\frac{1}{2}\left(-\frac{7}{6}\right)=\frac{37}{12},
\qquad
F_1(3)=\frac{11}{3}+\frac{1}{2}\left(\frac{7}{3}\right)=\frac{29}{6}.
$$

The predictions move toward their targets, but only partway because of shrinkage. The initial half-squared-error is $13/3$; after this step it is $61/48$. This example shows how fitting a tree to residuals reduces squared error locally without claiming that every future step always improves validation performance.

## The Additive Model and Gradient Step

Gradient Boosting builds an additive predictor. At round $m$, it adds a small contribution from a new learner $h_m$:

$$
F_m(x)=F_{m-1}(x)+\eta h_m(x)
$$

Here $F_{m-1}$ is the current prediction function and $\eta$ is the learning rate. The loss for the new model is evaluated at the updated predictions, not by directly averaging independent models.

For training examples $(x_i,y_i)$ and a differentiable loss $L(y,F(x))$, the next learner is fit to the negative gradient of the loss with respect to the current prediction:

$$
r_{im}=-\left[\frac{\partial L(y_i,F(x_i))}{\partial F(x_i)}\right]_{F=F_{m-1}}
$$

These values are called pseudo-residuals. The model takes a step in a function-space direction that locally reduces the chosen loss. For squared-error regression, the negative gradient is the ordinary residual $y_i-F_{m-1}(x_i)$; for other losses it is generally different.

## Learning Rate, Number of Learners, and Tree Size

The learning rate $\eta$ shrinks each learner's contribution. A smaller value makes each update more conservative and often requires more boosting rounds to reach similar training loss. It can improve generalization, but does not guarantee it.

The number of learners controls how many sequential updates are added. The depth or leaf count of each tree controls how complex a single update can be. Shallow trees capture simple interactions; deeper trees can fit more complex patterns but may overfit more quickly.

These choices interact. Tune the learning rate, number of rounds, and tree complexity together using validation data. Early stopping can select a round based on validation performance; keep the test set separate for final evaluation.

## Classification and Other Losses

Gradient Boosting can optimize classification losses by fitting pseudo-residuals rather than treating labels as numeric residual targets. For binary classification with logistic loss, let $F(x)$ be a raw score and $p(x)=\sigma(F(x))=1/(1+e^{-F(x)})$ the predicted probability. The negative gradient with respect to the raw score is:

$$
y_i-p_i
$$

A tree is fitted to these values. The update to its leaf scores is determined by reducing logistic loss; simply adding the pseudo-residual as though it were an ordinary class label is not the full procedure. The raw score is converted to a probability through the sigmoid function, and a classification threshold is applied separately.

For multiclass classification, implementations use vector-valued scores and a softmax-related loss. Other differentiable losses can support objectives such as absolute-error-like regression or ranking, though the exact algorithm and supported losses depend on the implementation.

## Worked Classification Gradient and Tree Update

Use binary logistic loss with labels $y\in\{0,1\}$ and raw score $F$. The predicted probability is $p=\sigma(F)$, and the negative gradient with respect to $F$ is $y-p$.

Take two observations with feature values $x=(0,1)$ and labels $y=(0,1)$. Start with raw score $F_0=0$ for both, so each has $p=0.5$. Their negative gradients are $-0.5$ and $+0.5$. A stump splitting at $x\leq0.5$ puts one observation in each leaf, so fitting the leaf values to the pseudo-residuals gives $h(x)=(-0.5,+0.5)$.

With learning rate $\eta=0.4$, update the raw scores:

$$
F_1(0)=0+0.4(-0.5)=-0.2,
\qquad
F_1(1)=0+0.4(0.5)=0.2.
$$

The updated probabilities are $\sigma(-0.2)\approx0.450$ and $\sigma(0.2)\approx0.550$. The average logistic loss falls from $\ln 2\approx0.693$ to $\ln(1+e^{-0.2})\approx0.598$. Here the leaf values fit the negative gradients and the chosen learning rate shrinks the step; other implementations may choose leaf values by directly minimizing loss or using a Newton-style update.

## Strengths, Limitations, and Practical Use

Gradient Boosting is flexible: the loss defines the quantity being optimized, while weak learners such as shallow trees can represent nonlinear patterns and feature interactions. It is often effective on structured data and does not generally require feature scaling when the base learners are decision trees.

The sequential training process is less naturally parallel across boosting rounds than bagging. Many rounds of trees can be computationally expensive, and flexible learners can overfit. Noisy targets or outliers can exert strong influence depending on the loss. The model is also less directly interpretable than one small tree, and the best settings are data-dependent.

Choose an objective and evaluation metric that reflect the real task, compare against simple baselines, and tune tree complexity, learning rate, and number of rounds using validation. Early stopping helps control training length but must monitor a validation set, never the final test set. Check class imbalance, probability calibration, and error costs separately when doing classification.

Gradient Boosting is a general framework rather than one specific library. Implementations such as classic Gradient Boosting, XGBoost, LightGBM, and CatBoost use related additive ideas but differ in tree construction, regularization, categorical handling, and computational optimizations.